<a href="https://colab.research.google.com/github/mmilannaik/bostonhousepricing/blob/main/W14S2_SQL_Grouping_Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Sleep Efficiency Dataset
For questions 1-5, you can find the dataset and details about it from [here](https://www.kaggle.com/datasets/equilibriumm/sleep-efficiency).

`ID`
a unique identifier for each test subject

`Age`
age of the test subject

`Gender`
male or female

`Bedtime`
the time the test subject goes to bed each night

`Wakeup time`
the time the test subject wakes up each morning

`Sleep duration`
the total amount of time the test subject slept (in hours)

`Sleep efficiency`
a measure of the proportion of time in bed spent asleep

`REM sleep percentage`
the percentage of total sleep time spent in REM sleep

`Deep sleep percentage`
the percentage of total sleep time spent in deep sleep

`Light sleep percentage`
the percentage of total sleep time spent in light sleep

`Awakenings`
the number of times the test subject wakes up during the night

`Caffeine consumption`
the amount of caffeine consumed in the 24 hours prior to bedtime (in mg)

`Alcohol consumption`
the amount of alcohol consumed in the 24 hours prior to bedtime (in oz)

`Smoking status`
whether or not the test subject smokes

`Exercise frequency`
the number of times the test subject exercises each week



### **`Problem 1:`**

**The question is:**

Find out the average sleep duration of top 15 male candidates who's sleep duration are equal to 7.5 or greater than 7.5.


### `Problem 2:` Show avg deep sleep time for both gender. Round result at 2 decimal places.

Note: sleep time and deep sleep percentage will give you, deep sleep time.


### **`Problem 3:`**


**The question is:**

Find out the lowest 10th to 30th light sleep percentage records where deep sleep percentage values are between 25 to 45. Display age, light sleep percentage and deep sleep percentage columns only.



### `Problem 4:` Group by on exercise frequency and smoking status and show average deep sleep time, average light sleep time and avg rem sleep time.

* Note the differences in deep sleep time for smoking and non smoking status



### `Problem 5:` Group By on Awekning and show AVG Caffeine consumption, AVG Deep sleep time and AVG Alcohol consumption only for people who do exercise atleast 3 days a week. Show result in descending order awekenings

# configurations

In [1]:
# 1. Install the Kaggle CLI
!pip install kaggle --quiet

# 2. Upload your Kaggle API token
#    • On Kaggle: Account → Create New API Token → download kaggle.json
#    • In Colab:
from google.colab import files
files.upload()   # select your kaggle.json

# 3. Configure the CLI
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 4. Download & unzip the dataset
!kaggle datasets download -d equilibriumm/sleep-efficiency
!unzip -q sleep-efficiency.zip   # adjust if the zip has a folder

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/equilibriumm/sleep-efficiency
License(s): copyright-authors


In [2]:
# 5. Load it into pandas
import pandas as pd
df = pd.read_csv('/content/Sleep_Efficiency.csv')   # replace with the actual CSV name

In [3]:

# 1. Install pandasql
!pip install pandasql --quiet

# 2. Load libraries and your CSV into a pandas DataFrame
import pandas as pd
from pandasql import sqldf

# 3. Create a helper to run SQL against any DataFrame in your notebook
pysqldf = lambda query: sqldf(query, globals())

  Preparing metadata (setup.py) ... done


In [4]:
df.columns

Index(['ID', 'Age', 'Gender', 'Bedtime', 'Wakeup time', 'Sleep duration',
       'Sleep efficiency', 'REM sleep percentage', 'Deep sleep percentage',
       'Light sleep percentage', 'Awakenings', 'Caffeine consumption',
       'Alcohol consumption', 'Smoking status', 'Exercise frequency'],
      dtype='object')

In [8]:
df.head(3)

,ID,Age,Gender,Bedtime,Wakeup time,Sleep duration,Sleep efficiency,REM sleep percentage,Deep sleep percentage,Light sleep percentage,Awakenings,Caffeine consumption,Alcohol consumption,Smoking status,Exercise frequency
0,1,65,Female,2021-03-06 01:00:00,2021-03-06 07:00:00,6.0,0.88,18,70,12,0.0,0.0,0.0,Yes,3.0
1,2,69,Male,2021-12-05 02:00:00,2021-12-05 09:00:00,7.0,0.66,19,28,53,3.0,0.0,3.0,Yes,3.0
2,3,40,Female,2021-05-25 21:30:00,2021-05-25 05:30:00,8.0,0.89,20,70,10,1.0,0.0,0.0,No,3.0


# P1 : Find out the average sleep duration of top 15 male candidates who's sleep duration are equal to 7.5 or greater than 7.5.

In [11]:
pysqldf('''


SELECT  AVG([Sleep duration]) FROM df
WHERE Gender = 'Male' and [Sleep duration] >= 7.5
ORDER BY [Sleep duration] DESC
LIMIT 15
''')

,AVG([Sleep duration])
0,8.015873


# P2: Show avg deep sleep time for both gender. Round result at 2 decimal places.

In [13]:
pysqldf('''


SELECT  ROUND(AVG([Sleep duration]*[Deep sleep percentage]/100),2) AS 'Avg_deep_sleep_time' FROM df
''')

,Avg_deep_sleep_time
0,3.94


# P3: Find out the lowest 10th to 30th light sleep percentage records where deep sleep percentage values are between 25 to 45. Display age, light sleep percentage and deep sleep percentage columns only.

In [23]:
pysqldf('''
WITH PercentileData AS (
 SELECT  Age,[light sleep percentage],[Deep sleep percentage],
 PERCENT_RANK() OVER(ORDER BY [light sleep percentage]) AS LightSleepPercentile
 FROM df
 WHERE [Deep sleep percentage] BETWEEN 25 and 45
)

SELECT  Age,[light sleep percentage],[Deep sleep percentage]
FROM PercentileData
WHERE LightSleepPercentile BETWEEN 10 and 30
ORDER BY LightSleepPercentile
''')

,Age,light sleep percentage,Deep sleep percentage


In [19]:
df.head(2)

,ID,Age,Gender,Bedtime,Wakeup time,Sleep duration,Sleep efficiency,REM sleep percentage,Deep sleep percentage,Light sleep percentage,Awakenings,Caffeine consumption,Alcohol consumption,Smoking status,Exercise frequency
0,1,65,Female,2021-03-06 01:00:00,2021-03-06 07:00:00,6.0,0.88,18,70,12,0.0,0.0,0.0,Yes,3.0
1,2,69,Male,2021-12-05 02:00:00,2021-12-05 09:00:00,7.0,0.66,19,28,53,3.0,0.0,3.0,Yes,3.0


# P4: Group by on exercise frequency and smoking status and show average deep sleep time, average light sleep time and avg rem sleep time.
Note the differences in deep sleep time for smoking and non smoking status

In [21]:
pysqldf('''

SELECT [Exercise frequency],[Smoking status],
AVG([Sleep duration] * [Deep Sleep percentage]),
AVG([Sleep duration] * [Light Sleep percentage]),
AVG([Sleep duration] * [REM Sleep percentage])

FROM df
GROUP BY [Exercise frequency],[Smoking status]

''')

,Exercise frequency,Smoking status,AVG([Sleep duration] * [Deep Sleep percentage]),AVG([Sleep duration] * [Light Sleep percentage]),AVG([Sleep duration] * [REM Sleep percentage])
0,NaN,No,549.000000,124.250000,151.750000
1,NaN,Yes,325.500000,224.000000,150.500000
2,0.0,No,374.311765,216.600000,164.970588
3,0.0,Yes,336.661290,264.112903,168.580645
4,1.0,No,396.875000,163.089286,177.535714
5,1.0,Yes,368.085366,203.158537,178.756098
6,2.0,No,443.816667,141.416667,141.433333
7,2.0,Yes,388.812500,166.708333,167.395833
8,3.0,No,431.835526,146.276316,179.782895
9,3.0,Yes,358.157407,218.064815,168.222222


# P5.Group By on Awekning and show AVG Caffeine consumption, AVG Deep sleep time and AVG Alcohol consumption only for people who do exercise atleast 3 days a week. Show result in descending order awekenings

In [28]:
df.columns

Index(['ID', 'Age', 'Gender', 'Bedtime', 'Wakeup time', 'Sleep duration',
       'Sleep efficiency', 'REM sleep percentage', 'Deep sleep percentage',
       'Light sleep percentage', 'Awakenings', 'Caffeine consumption',
       'Alcohol consumption', 'Smoking status', 'Exercise frequency'],
      dtype='object')

In [30]:
pysqldf('''
SELECT Awakenings,AVG([Caffeine consumption]),AVG([Alcohol consumption]),
AVG([Deep sleep percentage]*[Sleep duration])
FROM df
WHERE [Exercise frequency] >= 3
GROUP BY Awakenings
ORDER BY Awakenings DESC

''')

,Awakenings,AVG([Caffeine consumption]),AVG([Alcohol consumption]),AVG([Deep sleep percentage]*[Sleep duration])
0,4.0,7.352941,1.312500,357.088235
1,3.0,3.125000,1.823529,340.970588
2,2.0,8.333333,1.333333,374.235294
3,1.0,14.492754,1.285714,420.083333
4,0.0,27.222222,0.581395,442.266667
5,NaN,27.272727,1.818182,416.681818


## Power Generation Dataset

For this question, you can find the details as well as the dataset from [here](https://www.kaggle.com/datasets/arvindnagaonkar/power-generation-data).

In [32]:
#Download & unzip the dataset
!kaggle datasets download -d arvindnagaonkar/power-generation-data
!unzip -q power-generation-data.zip

Dataset URL: https://www.kaggle.com/datasets/arvindnagaonkar/power-generation-data
License(s): CC-BY-SA-4.0


In [33]:
power = pd.read_csv('/content/PowerGeneration.csv')

In [34]:
power.columns

Index(['Date', 'Power Station', 'Monitored Cap.(MW)',
       'Total Cap. Under Maintenace (MW)', 'Planned Maintanence (MW)',
       'Forced Maintanence(MW)', 'Other Reasons (MW)', 'Programme (MW)',
       'Actual (MW)', 'Excess(+) / Shortfall (-) (MW)', 'Deviation (MW)'],
      dtype='object')

In [36]:
power.head(3)

,Date,Power Station,Monitored Cap.(MW),Total Cap. Under Maintenace (MW),Planned Maintanence (MW),Forced Maintanence(MW),Other Reasons (MW),Programme (MW),Actual (MW),Excess(+) / Shortfall (-) (MW),Deviation (MW)
0,01-09-2017,Delhi,2235.4,135.0,0.0,135.0,0.0,13.29,18.29,5.0,37.62
1,01-09-2017,STPL,1350.0,1350.0,0.0,1350.0,0.0,0.00,0.00,0.0,0.00
2,01-09-2017,SPPL,150.0,150.0,0.0,150.0,0.0,0.00,0.00,0.0,0.00


### **`Problem 6:`**



**The question is:**

Display those power stations which have average 'Monitored Cap.(MW)' (display the values) between 1000 and 2000 and the number of occurance of the power stations (also display these values) are
greater than 200. Also sort the result in ascending order.




In [38]:
pysqldf('''
SELECT [Power Station],AVG([Monitored Cap.(MW)]),count([Power Station]) AS 'No_of_stations'
FROM power
WHERE [Monitored Cap.(MW)] BETWEEN 1000 and 2000
GROUP BY [Power Station]
HAVING count([Power Station]) > 200
ORDER BY count([Power Station]) ASC

''')

,Power Station,AVG([Monitored Cap.(MW)]),No_of_stations
0,SEIL,1320.000000,277
1,BRBCL,1000.000000,431
2,SGPL,1320.000000,442
3,JSWBL,1080.000000,446
4,NPGCL,1626.132075,636
5,MUNPL,1320.000000,701
6,TATA PCL,1430.000000,737
7,0,1399.200976,820
8,TOR. POW. (UNOSUGEN),1944.500000,1127
9,NEEPCO.,1487.455319,1175


## Avg Cost of Undergrad College by State(USA) Dataset-
For this question, you can find the detailed dataset from [here](https://www.kaggle.com/datasets/kfoster150/avg-cost-of-undergrad-college-by-state).

`Year`
The Digest year this information comes from

`State`
The U.S. State

`Type`
Type of University, Private or Public and in-state or out-of-state. Private colleges charge the same for in/out of state

`Length`
Whether the college mainly offers 2-year (Associates) or 4-year (Bachelors) programs

`Expense`
The Expense being described, tuition/fees or on-campus living expenses

`Value`
The average cost for this particular expense, in USD ($)

In [41]:
#Download & unzip the dataset
!kaggle datasets download -d kfoster150/avg-cost-of-undergrad-college-by-state
!unzip -q avg-cost-of-undergrad-college-by-state.zip

Dataset URL: https://www.kaggle.com/datasets/kfoster150/avg-cost-of-undergrad-college-by-state
License(s): CC0-1.0
avg-cost-of-undergrad-college-by-state.zip: Skipping, found more recently modified local copy (use --force to force download)
replace nces330_20.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y


In [42]:
grad = pd.read_csv('/content/nces330_20.csv')

In [43]:
grad.columns

Index(['Year', 'State', 'Type', 'Length', 'Expense', 'Value'], dtype='object')

In [44]:
grad.head(3)

,Year,State,Type,Length,Expense,Value
0,2013,Alabama,Private,4-year,Fees/Tuition,13983
1,2013,Alabama,Private,4-year,Room/Board,8503
2,2013,Alabama,Public In-State,2-year,Fees/Tuition,4048


### **`Problem 7:`**



**The question is:**

Display top 10 lowest "value" State names of which the Year either belong to 2013 or 2017 or 2021 and type is 'Public In-State'. Also the number of occurance should be between 6 to 10. Display the average value upto 2 decimal places, state names and the occurance of the states.


###`Problem -8:` Best state in terms of low education cost (Tution Fees) in 'Public' type university.



### `Problem 9:` 2nd Costliest state for Private education in year 2021. Consider, Tution and Room fee both.

In [54]:
grad['Type'].unique()

array(['Private', 'Public In-State', 'Public Out-of-State'], dtype=object)

In [57]:
grad['Expense'].unique()

array(['Fees/Tuition', 'Room/Board'], dtype=object)

# P.7

In [50]:
pysqldf('''
SELECT [State],AVG([Value]),COUNT([State])
FROM grad
WHERE YEAR IN (2013,2017,2021) AND TYPE = 'Public In-State'
GROUP BY State
HAVING COUNT(State) BETWEEN 6 AND 10
ORDER BY AVG(Value) ASC
LIMIT 10

''')

,State,AVG([Value]),COUNT([State])
0,New Mexico,5403.500,8
1,Utah,5565.500,8
2,Wyoming,5753.250,8
3,Idaho,5805.375,8
4,Florida,5940.375,8
5,Oklahoma,6088.000,8
6,North Carolina,6103.000,8
7,Montana,6158.125,8
8,Mississippi,6200.375,8
9,Arkansas,6239.000,8


# P.8

In [56]:
pysqldf('''
SELECT State
FROM grad
WHERE Type LIKE 'Public%' AND Expense = 'Fees/Tuition'
GROUP BY State
ORDER BY SUM(Value) ASC
LIMIT 1
''')

,State
0,District of Columbia


# P9

In [61]:
pysqldf('''
SELECT State
FROM grad
WHERE Type = 'Private' AND Year = 2021
GROUP BY State
ORDER BY SUM(Value) DESC
LIMIT 1,1
''')

,State
0,Vermont


### **`Problem 10:`**

For this, you can find the dataset from [here]().

**The question is:**

Display total and average values of Discount_offered for all the combinations of 'Mode_of_Shipment' (display this feature) and 'Warehouse_block' (display this feature also) for all male ('M') and 'High' Product_importance. Also sort the values in descending order of Mode_of_Shipment and ascending order of Warehouse_block.

## Config

In [67]:
#Download & unzip the dataset
!kaggle datasets download -d ulrikthygepedersen/shipping-ecommerce
!unzip -q shipping-ecommerce.zip


Dataset URL: https://www.kaggle.com/datasets/ulrikthygepedersen/shipping-ecommerce
License(s): Attribution 4.0 International (CC BY 4.0)
shipping-ecommerce.zip: Skipping, found more recently modified local copy (use --force to force download)


In [68]:
ecom = pd.read_csv('/content/shipping_ecommerce.csv')

In [69]:
ecom.shape

(10998, 10)

In [70]:
ecom.columns

Index(['Customer_care_calls', 'Customer_rating', 'Prior_purchases',
       'Discount_offered', 'Weight_in_gms', 'Warehouse_block',
       'Mode_of_Shipment', 'Product_importance', 'Gender', 'Class'],
      dtype='object')

In [71]:
ecom.head(2)

,Customer_care_calls,Customer_rating,Prior_purchases,Discount_offered,Weight_in_gms,Warehouse_block,Mode_of_Shipment,Product_importance,Gender,Class
0,5,4,2,10,5395,A,Ship,medium,M,1
1,4,3,2,6,5867,F,Ship,medium,F,0


In [72]:
ecom['Product_importance'].unique()

array(['medium', 'low', 'high'], dtype=object)

## Q10

In [75]:
pysqldf('''
SELECT Mode_of_Shipment,Warehouse_block,
AVG(Discount_offered),SUM(Discount_offered)
FROM ecom
WHERE Gender = 'M' AND Product_importance = 'high'
GROUP BY Mode_of_Shipment,Warehouse_block
ORDER BY Mode_of_Shipment DESC,Warehouse_block ASC
''')

,Mode_of_Shipment,Warehouse_block,AVG(Discount_offered),SUM(Discount_offered)
0,Ship,A,16.023256,689
1,Ship,B,18.255814,785
2,Ship,C,11.346939,556
3,Ship,D,12.344262,753
4,Ship,F,15.050000,1505
5,Road,A,18.600000,279
6,Road,B,11.833333,142
7,Road,C,9.857143,138
8,Road,D,18.777778,338
9,Road,F,13.461538,350
